In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
from scipy import stats

# ==========================
# 1. LOAD ALL FILES FROM ZIP
# ==========================

zip_path = "d2c churn data package.zip"

dataframes = {}

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("extracted_data")

for root, dirs, files in os.walk("extracted_data"):
    for file in files:
        if file.endswith(".csv"):
            file_path = os.path.join(root, file)

            try:
                df = pd.read_csv(file_path)
                dataframes[file] = df
                print(f"Loaded: {file} | Shape: {df.shape}")
            except Exception as e:
                print(f"Error loading {file}: {e}")

# ==========================
# 2. DATA QUALITY FUNCTION
# ==========================

def data_quality_report(df, name):

    report = {}

    report["Dataset"] = name
    report["Rows"] = df.shape[0]
    report["Columns"] = df.shape[1]

    # Missing Values
    missing = df.isnull().sum()
    report["Missing Values"] = missing[missing > 0].to_dict()

    # Duplicate Records
    report["Duplicate Rows"] = df.duplicated().sum()

    # Data Types
    report["Data Types"] = df.dtypes.astype(str).to_dict()

    # Numeric Outliers (IQR Method)
    numeric_cols = df.select_dtypes(include=np.number).columns

    outlier_counts = {}

    for col in numeric_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = ((df[col] < lower) | (df[col] > upper)).sum()

        if outliers > 0:
            outlier_counts[col] = int(outliers)

    report["Outliers"] = outlier_counts

    return report

# ==========================
# 3. GENERATE REPORTS
# ==========================

all_reports = []

for name, df in dataframes.items():

    report = data_quality_report(df, name)

    all_reports.append(report)

# Display Summary

for rpt in all_reports:

    print("\n" + "="*80)
    print("DATASET:", rpt["Dataset"])
    print("="*80)

    print("Rows:", rpt["Rows"])
    print("Columns:", rpt["Columns"])

    print("\nMissing Values:")
    print(rpt["Missing Values"])

    print("\nDuplicate Rows:")
    print(rpt["Duplicate Rows"])

    print("\nOutliers:")
    print(rpt["Outliers"])

# ==========================
# 4. SAVE REPORT TO EXCEL
# ==========================

summary_rows = []

for rpt in all_reports:

    summary_rows.append({
        "Dataset": rpt["Dataset"],
        "Rows": rpt["Rows"],
        "Columns": rpt["Columns"],
        "Duplicate Rows": rpt["Duplicate Rows"],
        "Missing Columns": len(rpt["Missing Values"]),
        "Outlier Columns": len(rpt["Outliers"])
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_excel(
    "Data_Quality_Summary.xlsx",
    index=False
)

print("\nData Quality Report saved.")